# 04 â€” Prompt Formatting

**Project:** Ecommerce-LLM-Finetuning
**Goal of this notebook:**
1. Load cleaned instruction-response pairs from notebook 03
2. Convert them into the project's instruction-tuning format:

```
### Instruction:
Customer:
<question>

### Response:
<answer>
```

3. Tokenize with the target model's tokenizer and inspect resulting lengths
4. Split into train/val/test and save as Hugging Face-ready JSONL

This uses `src/prompt_template.py::PromptFormatter` so notebook, training,
and inference code all share one prompt-building implementation.


In [1]:
!pip install -q transformers sentencepiece datasets


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Program Files\Python313\Scripts\pip.exe\__main__.py", line 2, in <module>
  File "C:\Users\Asus\AppData\Roaming\uv\python\cpython-3.10-windows-x86_64-none\Lib\re.py", line 125, in <module>
    import sre_compile
  File "C:\Users\Asus\AppData\Roaming\uv\python\cpython-3.10-windows-x86_64-none\Lib\sre_compile.py", line 17, in <module>
    assert _sre.MAGIC == MAGIC, "SRE module mismatch"
AssertionError: SRE module mismatch


In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from transformers import AutoTokenizer

from src.config import Config
from src.prompt_template import PromptFormatter
from src.data_loader import EcommerceDataLoader
from src.utils import get_logger

logger = get_logger("notebook04")
cfg = Config()
cfg.ensure_dirs()


c:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load cleaned pairs

In [3]:
clean_path = cfg.processed_data_dir / "cleaned_pairs.jsonl"
df = pd.read_json(clean_path, lines=True)
print(f"Loaded {len(df):,} cleaned pairs")
df.head(3)


Loaded 26,867 cleaned pairs


,instruction,response,category
0,question about cancelling order Order Number,I've understood you have a question regarding ...,ORDER
1,i have a question about cancelling oorder Orde...,I've been informed that you have a question ab...,ORDER
2,i need help cancelling puchase Order Number,I can sense that you're seeking assistance wit...,ORDER


## 2. Build the instruction-formatted `text` column

In [4]:
formatter = PromptFormatter(cfg)
df_formatted = formatter.add_text_column(df, style="instruction")

print(df_formatted.iloc[0]["text"])
print("\n" + "=" * 60 + "\n")
print(df_formatted.iloc[1]["text"])


### Instruction:
Customer:
question about cancelling order Order Number

### Response:
I've understood you have a question regarding canceling order Order Number , and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


### Instruction:
Customer:
i have a question about cancelling oorder Order Number

### Response:
I've been informed that you have a question about canceling order Order Number . I'm here to assist you! Please go ahead and let me know what specific question you have, and I'll provide you with all the information and guidance you need. Your satisfaction is my top priority.


## 3. Tokenize and inspect lengths

We use the same tokenizer that will be used for fine-tuning so `max_seq_length` in `Config` is well-calibrated.

In [5]:
try:
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_name)
except Exception as e:
    logger.warning(f"Falling back to a generic instruct tokenizer for EDA: {e}")
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

df_formatted = formatter.tokenize_lengths(df_formatted, tokenizer, text_col="text")
print(df_formatted["token_length"].describe())

pct_over = (df_formatted["token_length"] > cfg.max_seq_length).mean() * 100
print(f"\n% of examples exceeding max_seq_length ({cfg.max_seq_length}): {pct_over:.2f}%")


count    26867.000000
mean       141.590762
std         64.240370
min         28.000000
25%        100.000000
50%        122.000000
75%        168.000000
max        492.000000
Name: token_length, dtype: float64

% of examples exceeding max_seq_length (2048): 0.00%


## 4. (Optional) Preview chat-template formatting

For Llama-3.2/Qwen2.5-Instruct models, `apply_chat_template` is the canonical way to format multi-turn/system-prompted data. We show it here for reference; the SFTTrainer in notebook 05 uses the raw `text` column built above for simplicity and full control over the prompt format.

In [6]:
messages = formatter.format_chat_messages(df.iloc[0]["instruction"], df.iloc[0]["response"])
print(messages)

if getattr(tokenizer, "chat_template", None):
    rendered = tokenizer.apply_chat_template(messages, tokenize=False)
    print("\n--- Rendered chat template ---\n")
    print(rendered)
else:
    print("\nTokenizer has no chat_template attribute; using raw instruction format instead.")


[{'role': 'system', 'content': 'You are a helpful, polite Ecommerce Customer Support Assistant. You answer questions about orders, shipping, refunds, returns, payments, coupons, delivery, and account management. If a question is unrelated to ecommerce customer support, politely say that you can only help with ecommerce support questions.'}, {'role': 'user', 'content': 'question about cancelling order Order Number'}, {'role': 'assistant', 'content': "I've understood you have a question regarding canceling order Order Number , and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}]

--- Rendered chat template ---

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 02 Jul 2026

You are a helpful, polite Ecommerce Customer Support Assistant. You answer questions about orders, shipping, refunds, returns, payments, coupons, delivery, and account manageme

## 5. Split into train / val / test and save

In [7]:
loader = EcommerceDataLoader(cfg)
train_df, val_df, test_df = loader.split_dataset(
    df_formatted[["instruction", "response", "text"]],
    train_ratio=cfg.train_split_ratio,
    val_ratio=cfg.val_split_ratio,
    seed=cfg.seed,
)
loader.save_splits(train_df, val_df, test_df)

print(f"train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
train_df.head(2)


13:05:46 | INFO     | src.data_loader | Split sizes -> train: 21493, val: 2686, test: 2688
13:05:47 | INFO     | src.utils | Saved 21493 records -> C:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\processed\train.jsonl
13:05:47 | INFO     | src.utils | Saved 2686 records -> C:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\processed\val.jsonl
13:05:47 | INFO     | src.utils | Saved 2688 records -> C:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\processed\test.jsonl
train: 21493, val: 2686, test: 2688


,instruction,response,text
0,how do I check what hours I can call customer ...,Grateful for your contact! I get the sense tha...,### Instruction:\nCustomer:\nhow do I check wh...
1,switch to gold account,Assuredly! I'm thrilled to guide you through t...,### Instruction:\nCustomer:\nswitch to gold ac...


## Summary

- Converted cleaned pairs into `### Instruction: / ### Response:` formatted training strings
- Verified token lengths comfortably fit `max_seq_length`
- Previewed the chat-template alternative for reference
- Saved final `train.jsonl` / `val.jsonl` / `test.jsonl` to `data/processed/`

Next: `05_finetune_llm.ipynb` â€” the main QLoRA fine-tuning notebook.